## Dataset 2: ACSEmployment 2018 (California to Texas Geographic Shift)

In [1]:
import sys
!{sys.executable} -m pip install folktables --break-system-packages

In [2]:
import pandas as pd
import numpy as np
from folktables import ACSDataSource, ACSEmployment
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

output_dir = './thesis_figures_acs/'
os.makedirs(output_dir, exist_ok=True)

SEEDS = [42, 43, 44]
metrics = ['AUROC', 'AUPRC', 'Brier Score']
all_results = []

## Dataset

The ACSEmployment task from the `folktables` library frames binary classification: predicting whether an individual is employed, using 16 numeric ACS survey features. 

California (CA) 2018 data serves as the in-domain set, split 70% train / 15% validation / 15% test, while all Texas (TX) 2018 rows are held out as the shifted test set to simulate a natural geographic distribution shift.

In [4]:
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')

acs_ca = data_source.get_data(states=['CA'], download=True)

acs_tx = data_source.get_data(states=['TX'], download=True)

X_ca_np, y_ca_np, _ = ACSEmployment.df_to_numpy(acs_ca)
X_tx_np, y_tx_np, _ = ACSEmployment.df_to_numpy(acs_tx)

df_ca = pd.DataFrame(X_ca_np, columns=ACSEmployment.features)
df_ca['y'] = y_ca_np.astype(int)

df_tx = pd.DataFrame(X_tx_np, columns=ACSEmployment.features)
df_tx['y'] = y_tx_np.astype(int)

print(f"CA shape: {df_ca.shape}  |  TX shape: {df_tx.shape}")
print(f"CA label balance: {df_ca['y'].mean():.3f}  |  TX label balance: {df_tx['y'].mean():.3f}")
df_ca.head()

CA shape: (378817, 17)  |  TX shape: (268100, 17)
CA label balance: 0.456  |  TX label balance: 0.452


,AGEP,SCHL,MAR,RELP,DIS,ESP,CIT,MIG,MIL,ANC,NATIVITY,DEAR,DEYE,DREM,SEX,RAC1P,y
0,30.0,14.0,1.0,16.0,2.0,0.0,1.0,3.0,4.0,1.0,1.0,2.0,2.0,2.0,1.0,8.0,0
1,18.0,14.0,5.0,17.0,2.0,0.0,1.0,1.0,4.0,1.0,1.0,2.0,2.0,2.0,2.0,1.0,0
2,69.0,17.0,1.0,17.0,1.0,0.0,1.0,1.0,2.0,2.0,1.0,2.0,2.0,2.0,1.0,9.0,0
3,25.0,1.0,5.0,17.0,1.0,0.0,1.0,1.0,4.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,0
4,31.0,18.0,5.0,16.0,2.0,0.0,1.0,1.0,4.0,1.0,1.0,2.0,2.0,2.0,2.0,1.0,0


## Data Splitting


Split is parameterised by `random_state` so each seed gives a reproducible but distinct split.

In [5]:
def data_split_function(df_ca, df_tx, random_state):
    """Split CA data into train/val/test; TX data is the shifted test set."""
    X_ca = df_ca.drop(columns=['y'])
    y_ca = df_ca['y']
    X_tx = df_tx.drop(columns=['y'])
    y_tx = df_tx['y']

    X_train, X_temp, y_train, y_temp = train_test_split(
        X_ca, y_ca, test_size=0.30, random_state=random_state, stratify=y_ca
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=random_state, stratify=y_temp
    )

    X_shifted, y_shifted = X_tx.reset_index(drop=True), y_tx.reset_index(drop=True)

    print(f"  Train: {X_train.shape}, Val: {X_val.shape}, Test ID: {X_test.shape}, Shifted: {X_shifted.shape}")
    return X_train, y_train, X_val, y_val, X_test, y_test, X_shifted, y_shifted

In [6]:
def evaluate_model(model, X_test, y_test):
    """Evaluate a trained model, returning AUROC, AUPRC, and Brier Score."""
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    auroc = roc_auc_score(y_test, y_pred_proba)
    auprc = average_precision_score(y_test, y_pred_proba)
    brier = brier_score_loss(y_test, y_pred_proba)
    return auroc, auprc, brier


def train_models(X_train, y_train, random_state):
    """Fit LR and LightGBM pipelines on numeric-only ACS features."""
    pipeline_lr = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler(with_mean=False)),
        ('classifier', LogisticRegression(
            max_iter=2000, solver='liblinear', class_weight='balanced', random_state=random_state
        ))
    ])
    pipeline_lgbm = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('classifier', LGBMClassifier(
            n_estimators=400, learning_rate=0.05, num_leaves=64,
            subsample=0.9, colsample_bytree=0.9,
            class_weight='balanced', random_state=random_state, verbose=-1
        ))
    ])
    pipeline_lr.fit(X_train, y_train)
    pipeline_lgbm.fit(X_train, y_train)
    return pipeline_lr, pipeline_lgbm


print('evaluate_model and train_models defined.')

evaluate_model and train_models defined.


## Data Perturbation Helpers

In [ ]:
def introduce_label_noise(y, noise_rate, random_state):
    """Introduce noise into training labels by flipping a fraction of them."""
    y_noisy = np.array(y, dtype=int).copy()
    if noise_rate == 0:
        return y_noisy
    rng = np.random.default_rng(random_state)
    n = len(y_noisy)
    k = int(round(noise_rate * n))
    idx = rng.choice(n, size=k, replace=False)
    y_noisy[idx] = 1 - y_noisy[idx]
    return y_noisy


def introduce_mcar_missingness(df, columns_to_modify, missing_rate, random_state):
    """Introduce Missing Completely At Random (MCAR) missingness."""
    df_mcar = df.copy()
    rng = np.random.default_rng(random_state)
    for col in columns_to_modify:
        if col in df_mcar.columns:
            non_nan_idx = df_mcar[col].dropna().index.to_numpy()
            num_nan = int(round(missing_rate * len(non_nan_idx)))
            if num_nan > 0:
                fill = None if df_mcar[col].dtype == object else np.nan
                df_mcar.loc[rng.choice(non_nan_idx, size=num_nan, replace=False), col] = fill
    return df_mcar


def introduce_mar_missingness(df, columns_to_modify, missing_rate, condition_fn, random_state):
    """Introduce Missing At Random (MAR) missingness for rows where condition_fn(df) is True."""
    df_mar = df.copy()
    rng = np.random.default_rng(random_state)
    cond_idx = df_mar[condition_fn(df_mar)].index
    if len(cond_idx) == 0 and missing_rate > 0:
        print(f"Warning: No rows match MAR condition. Skipping.")
        return df_mar
    for col in columns_to_modify:
        if col in df_mar.columns:
            target_idx = df_mar.loc[cond_idx, col].dropna().index.to_numpy()
            num_nan = int(round(missing_rate * len(target_idx)))
            if num_nan > 0:
                fill = None if df_mar[col].dtype == object else np.nan
                df_mar.loc[rng.choice(target_idx, size=num_nan, replace=False), col] = fill
    return df_mar


def blank_out_features(df_input, cols_to_blank):
    """Set specified feature columns to missing (simulate feature unavailability)."""
    df_blanked = df_input.copy()
    for col in cols_to_blank:
        if col in df_blanked.columns:
            if df_blanked[col].dtype == object:
                df_blanked[col] = None
            else:
                df_blanked[col] = np.nan
    return df_blanked


columns_for_missingness = ['AGEP', 'SCHL', 'MAR', 'DIS', 'CIT', 'NATIVITY', 'SEX', 'RAC1P']

feature_groups = {
    'Education':                    ['SCHL'],
    'Disability':                   ['DIS', 'DEAR', 'DEYE', 'DREM'],
    'Citizenship_Nativity_Ancestry': ['CIT', 'NATIVITY', 'ANC'],
}

print(f"columns_for_missingness: {columns_for_missingness}")
for name, cols in feature_groups.items():
    print(f"  {name}: {cols}")

columns_for_missingness: ['AGEP', 'SCHL', 'MAR', 'DIS', 'CIT', 'NATIVITY', 'SEX', 'RAC1P']
  Education: ['SCHL']
  Disability: ['DIS', 'DEAR', 'DEYE', 'DREM']
  Citizenship_Nativity_Ancestry: ['CIT', 'NATIVITY', 'ANC']


In [8]:
all_results = []  

## Baseline Experiment

Both models are trained on **clean California (CA) data** with no perturbations. Results on the in-domain CA test set and the shifted Texas (TX) test set establish the performance reference for all subsequent comparisons.

The baseline also trains the models that are reused in feature-availability shift, where blanking is applied only at evaluation time.

In [9]:
def run_baseline_experiment(seed, X_train, y_train, X_test, y_test, X_shifted, y_shifted):
    """Baseline: train on clean CA data, evaluate in-domain and shifted (TX)."""
    print(f"--- Baseline | Seed {seed} ---")
    pipeline_lr, pipeline_lgbm = train_models(X_train, y_train, seed)

    for model_name, pipeline in [('Logistic Regression', pipeline_lr), ('LightGBM', pipeline_lgbm)]:
        auroc_id, auprc_id, brier_id = evaluate_model(pipeline, X_test, y_test)
        auroc_sh, auprc_sh, brier_sh = evaluate_model(pipeline, X_shifted, y_shifted)
        base = {'Experiment': 'Baseline', 'Condition': 'None', 'Noise Rate': 0,
                'Missingness Type': 'None', 'Missing Rate': 0, 'Feature Group Blanked': 'None',
                'Model': model_name, 'Random Seed': seed}
        all_results.append({**base, 'Test Set': 'In-domain Test',
                             'AUROC': auroc_id, 'AUPRC': auprc_id, 'Brier Score': brier_id})
        all_results.append({**base, 'Test Set': 'Shifted Test',
                             'AUROC': auroc_sh, 'AUPRC': auprc_sh, 'Brier Score': brier_sh})

    return pipeline_lr, pipeline_lgbm

In [10]:
baseline_pipelines = {} 

for seed in SEEDS:
    print(f"\n===== Baseline | Seed {seed} =====")
    X_train, y_train, X_val, y_val, X_test, y_test, X_shifted, y_shifted = \
        data_split_function(df_ca, df_tx, seed)
    pipeline_lr, pipeline_lgbm = run_baseline_experiment(
        seed, X_train, y_train, X_test, y_test, X_shifted, y_shifted)
    baseline_pipelines[seed] = (pipeline_lr, pipeline_lgbm, X_test, y_test, X_shifted, y_shifted)

# Mini-summary
_base_df = pd.DataFrame(all_results)
_base_df = _base_df[_base_df['Experiment'] == 'Baseline']
print("\nBaseline mean AUROC across seeds:")
print(_base_df.groupby(['Model', 'Test Set'])['AUROC'].mean().round(4).to_string())


===== Baseline | Seed 42 =====
  Train: (265171, 16), Val: (56823, 16), Test ID: (56823, 16), Shifted: (268100, 16)
--- Baseline | Seed 42 ---

===== Baseline | Seed 43 =====
  Train: (265171, 16), Val: (56823, 16), Test ID: (56823, 16), Shifted: (268100, 16)
--- Baseline | Seed 43 ---

===== Baseline | Seed 44 =====
  Train: (265171, 16), Val: (56823, 16), Test ID: (56823, 16), Shifted: (268100, 16)
--- Baseline | Seed 44 ---

Baseline mean AUROC across seeds:
Model                Test Set      
LightGBM             In-domain Test    0.9037
                     Shifted Test      0.9108
Logistic Regression  In-domain Test    0.8443
                     Shifted Test      0.8551


## Label Noise Experiment

Training labels are randomly flipped at rates of **0%, 5%, 10%, and 20%**. The 0% condition replicates the baseline (clean labels) and serves as an internal anchor. This section tests how sensitive each model's generalisation gap is to progressively corrupted supervision signals.

In [11]:
def run_label_noise_experiment(seed, X_train, y_train, X_test, y_test, X_shifted, y_shifted):
    """Label noise sensitivity: vary noise rate at 0%, 5%, 10%, 20%."""
    print(f"--- Label Noise | Seed {seed} ---")
    for noise_rate in [0.0, 0.05, 0.10, 0.20]:
        print(f"  Noise rate: {noise_rate*100:.0f}%")
        y_noisy = introduce_label_noise(y_train, noise_rate, random_state=seed)
        pipeline_lr, pipeline_lgbm = train_models(X_train, y_noisy, seed)

        for model_name, pipeline in [('Logistic Regression', pipeline_lr), ('LightGBM', pipeline_lgbm)]:
            auroc_id, auprc_id, brier_id = evaluate_model(pipeline, X_test, y_test)
            auroc_sh, auprc_sh, brier_sh = evaluate_model(pipeline, X_shifted, y_shifted)
            base = {'Experiment': 'Label Noise', 'Condition': f'{noise_rate*100:.0f}% Noise',
                    'Noise Rate': noise_rate, 'Missingness Type': 'None', 'Missing Rate': 0,
                    'Feature Group Blanked': 'None', 'Model': model_name, 'Random Seed': seed}
            all_results.append({**base, 'Test Set': 'In-domain Test',
                                 'AUROC': auroc_id, 'AUPRC': auprc_id, 'Brier Score': brier_id})
            all_results.append({**base, 'Test Set': 'Shifted Test',
                                 'AUROC': auroc_sh, 'AUPRC': auprc_sh, 'Brier Score': brier_sh})

In [12]:
_noise_start = len(all_results)

for seed in SEEDS:
    print(f"\n===== Label Noise | Seed {seed} =====")
    X_train, y_train, X_val, y_val, X_test, y_test, X_shifted, y_shifted = \
        data_split_function(df_ca, df_tx, seed)
    run_label_noise_experiment(seed, X_train, y_train, X_test, y_test, X_shifted, y_shifted)

# Mini-summary
_noise_df = pd.DataFrame(all_results[_noise_start:])
print("\nLabel Noise mean AUROC (Shifted Test) across seeds:")
print(_noise_df[_noise_df['Test Set'] == 'Shifted Test']
      .groupby(['Model', 'Condition'])['AUROC'].mean().round(4).to_string())


===== Label Noise | Seed 42 =====
  Train: (265171, 16), Val: (56823, 16), Test ID: (56823, 16), Shifted: (268100, 16)
--- Label Noise | Seed 42 ---
  Noise rate: 0%
  Noise rate: 5%
  Noise rate: 10%
  Noise rate: 20%

===== Label Noise | Seed 43 =====
  Train: (265171, 16), Val: (56823, 16), Test ID: (56823, 16), Shifted: (268100, 16)
--- Label Noise | Seed 43 ---
  Noise rate: 0%
  Noise rate: 5%
  Noise rate: 10%
  Noise rate: 20%

===== Label Noise | Seed 44 =====
  Train: (265171, 16), Val: (56823, 16), Test ID: (56823, 16), Shifted: (268100, 16)
--- Label Noise | Seed 44 ---
  Noise rate: 0%
  Noise rate: 5%
  Noise rate: 10%
  Noise rate: 20%

Label Noise mean AUROC (Shifted Test) across seeds:
Model                Condition
LightGBM             0% Noise     0.9108
                     10% Noise    0.9096
                     20% Noise    0.9081
                     5% Noise     0.9104
Logistic Regression  0% Noise     0.8551
                     10% Noise    0.8506
          

## Missingness Experiment

Two missingness mechanisms are injected into training features at rates of **0%, 10%, and 20%**:
- **MCAR** (Missing Completely At Random): values are blanked randomly, independent of any other feature.
- **MAR_Age_Older** (Missing At Random): values are blanked only for rows where `AGEP ≥ 65`, simulating a realistic data-collection gap for older respondents.

Missingness is applied to a broad set of ACS columns: `AGEP, SCHL, MAR, DIS, CIT, NATIVITY, SEX, RAC1P`.

In [13]:
def run_missingness_experiment(seed, X_train, y_train, X_test, y_test, X_shifted, y_shifted):
    """Missingness injection: MCAR and MAR_Age_Older (AGEP >= 65) at 0%, 10%, 20%."""
    print(f"--- Missingness | Seed {seed} ---")
    missing_rates = [0.0, 0.10, 0.20]
    missingness_scenarios = [
        {'type': 'MCAR', 'label': 'MCAR',          'condition_fn': None},
        {'type': 'MAR',  'label': 'MAR_Age_Older',  'condition_fn': lambda df: df['AGEP'] >= 65},
    ]

    for scenario in missingness_scenarios:
        label = scenario['label']
        for rate in missing_rates:
            print(f"  {label} @ {rate*100:.0f}%")
            if scenario['type'] == 'MCAR' or rate == 0:
                X_train_noisy = introduce_mcar_missingness(
                    X_train, columns_for_missingness, rate, seed
                ) if scenario['type'] == 'MCAR' else X_train.copy()
            else:
                X_train_noisy = introduce_mar_missingness(
                    X_train, columns_for_missingness, rate, scenario['condition_fn'], seed
                )

            pipeline_lr, pipeline_lgbm = train_models(X_train_noisy, y_train, seed)

            for model_name, pipeline in [('Logistic Regression', pipeline_lr), ('LightGBM', pipeline_lgbm)]:
                auroc_id, auprc_id, brier_id = evaluate_model(pipeline, X_test, y_test)
                auroc_sh, auprc_sh, brier_sh = evaluate_model(pipeline, X_shifted, y_shifted)
                base = {'Experiment': 'Missingness', 'Condition': label, 'Noise Rate': 0,
                        'Missingness Type': label, 'Missing Rate': rate, 'Feature Group Blanked': 'None',
                        'Model': model_name, 'Random Seed': seed}
                all_results.append({**base, 'Test Set': 'In-domain Test',
                                     'AUROC': auroc_id, 'AUPRC': auprc_id, 'Brier Score': brier_id})
                all_results.append({**base, 'Test Set': 'Shifted Test',
                                     'AUROC': auroc_sh, 'AUPRC': auprc_sh, 'Brier Score': brier_sh})

In [14]:
_miss_start = len(all_results)

for seed in SEEDS:
    print(f"\n===== Missingness | Seed {seed} =====")
    X_train, y_train, X_val, y_val, X_test, y_test, X_shifted, y_shifted = \
        data_split_function(df_ca, df_tx, seed)
    run_missingness_experiment(seed, X_train, y_train, X_test, y_test, X_shifted, y_shifted)


_miss_df = pd.DataFrame(all_results[_miss_start:])
print("\nMissingness mean AUROC (Shifted Test) across seeds:")
print(_miss_df[_miss_df['Test Set'] == 'Shifted Test']
      .groupby(['Model', 'Missingness Type', 'Missing Rate'])['AUROC'].mean().round(4).to_string())


===== Missingness | Seed 42 =====
  Train: (265171, 16), Val: (56823, 16), Test ID: (56823, 16), Shifted: (268100, 16)
--- Missingness | Seed 42 ---
  MCAR @ 0%
  MCAR @ 10%
  MCAR @ 20%
  MAR_Age_Older @ 0%
  MAR_Age_Older @ 10%
  MAR_Age_Older @ 20%

===== Missingness | Seed 43 =====
  Train: (265171, 16), Val: (56823, 16), Test ID: (56823, 16), Shifted: (268100, 16)
--- Missingness | Seed 43 ---
  MCAR @ 0%
  MCAR @ 10%
  MCAR @ 20%
  MAR_Age_Older @ 0%
  MAR_Age_Older @ 10%
  MAR_Age_Older @ 20%

===== Missingness | Seed 44 =====
  Train: (265171, 16), Val: (56823, 16), Test ID: (56823, 16), Shifted: (268100, 16)
--- Missingness | Seed 44 ---
  MCAR @ 0%
  MCAR @ 10%
  MCAR @ 20%
  MAR_Age_Older @ 0%
  MAR_Age_Older @ 10%
  MAR_Age_Older @ 20%

Missingness mean AUROC (Shifted Test) across seeds:
Model                Missingness Type  Missing Rate
LightGBM             MAR_Age_Older     0.0             0.9108
                                       0.1             0.9095
            

## Feature-Availability Shift Experiment

Three ACS feature groups are blanked entirely **at evaluation time** (both in-domain and shifted test sets). Models are trained on complete, clean data; the blanking simulates a deployment scenario where certain data sources become unavailable.

Group and Columns blanked

Education: SCHL

Disability: DIS, DEAR, DEYE, DREM

Citizenship / Nativity / Ancestry: CIT, NATIVITY, ANC

The baseline-trained models from Section A are reused here — no retraining is needed.

In [15]:
def run_feature_shift_experiment(seed, X_test, y_test, X_shifted, y_shifted,
                                  pipeline_lr, pipeline_lgbm):
    """Feature shift: blank out feature groups at test time using baseline-trained models."""
    print(f"--- Feature Shift | Seed {seed} ---")
    for group_name, cols_to_blank in feature_groups.items():
        existing_cols = [c for c in cols_to_blank if c in X_test.columns]
        if not existing_cols:
            print(f"  Skipping {group_name}: no columns found")
            continue
        print(f"  Blanking: {group_name} ({existing_cols})")
        X_test_blanked    = blank_out_features(X_test, existing_cols)
        X_shifted_blanked = blank_out_features(X_shifted, existing_cols)

        for model_name, pipeline in [('Logistic Regression', pipeline_lr), ('LightGBM', pipeline_lgbm)]:
            auroc_id, auprc_id, brier_id = evaluate_model(pipeline, X_test_blanked, y_test)
            auroc_sh, auprc_sh, brier_sh = evaluate_model(pipeline, X_shifted_blanked, y_shifted)
            base = {'Experiment': 'Feature Shift', 'Condition': group_name, 'Noise Rate': 0,
                    'Missingness Type': 'None', 'Missing Rate': 0, 'Feature Group Blanked': group_name,
                    'Model': model_name, 'Random Seed': seed}
            all_results.append({**base, 'Test Set': 'In-domain Test',
                                 'AUROC': auroc_id, 'AUPRC': auprc_id, 'Brier Score': brier_id})
            all_results.append({**base, 'Test Set': 'Shifted Test',
                                 'AUROC': auroc_sh, 'AUPRC': auprc_sh, 'Brier Score': brier_sh})

In [16]:
_feat_start = len(all_results)

for seed in SEEDS:
    print(f"\n===== Feature Shift | Seed {seed} =====")
    pipeline_lr, pipeline_lgbm, X_test, y_test, X_shifted, y_shifted = baseline_pipelines[seed]
    run_feature_shift_experiment(seed, X_test, y_test, X_shifted, y_shifted, pipeline_lr, pipeline_lgbm)

# Mini-summary
_feat_df = pd.DataFrame(all_results[_feat_start:])
print("\nFeature Shift mean AUROC (Shifted Test) across seeds:")
print(_feat_df[_feat_df['Test Set'] == 'Shifted Test']
      .groupby(['Model', 'Feature Group Blanked'])['AUROC'].mean().round(4).to_string())

print("\n\nAll experiments complete.")


===== Feature Shift | Seed 42 =====
--- Feature Shift | Seed 42 ---
  Blanking: Education (['SCHL'])
  Blanking: Disability (['DIS', 'DEAR', 'DEYE', 'DREM'])
  Blanking: Citizenship_Nativity_Ancestry (['CIT', 'NATIVITY', 'ANC'])

===== Feature Shift | Seed 43 =====
--- Feature Shift | Seed 43 ---
  Blanking: Education (['SCHL'])
  Blanking: Disability (['DIS', 'DEAR', 'DEYE', 'DREM'])
  Blanking: Citizenship_Nativity_Ancestry (['CIT', 'NATIVITY', 'ANC'])

===== Feature Shift | Seed 44 =====
--- Feature Shift | Seed 44 ---
  Blanking: Education (['SCHL'])
  Blanking: Disability (['DIS', 'DEAR', 'DEYE', 'DREM'])
  Blanking: Citizenship_Nativity_Ancestry (['CIT', 'NATIVITY', 'ANC'])

Feature Shift mean AUROC (Shifted Test) across seeds:
Model                Feature Group Blanked        
LightGBM             Citizenship_Nativity_Ancestry    0.9081
                     Disability                       0.9002
                     Education                        0.9055
Logistic Regression  

## Aggregate and Summarize Results

1. Build `results_df` from raw results
2. Compute the generalization gap (Shifted − In-domain) per run and append to `results_df`
3. Build `summary_df` with mean ± std across seeds

In [17]:
results_df = pd.DataFrame(all_results)
print(f"Raw results shape: {results_df.shape}")
results_df.head()

Raw results shape: (168, 12)


,Experiment,Condition,Noise Rate,Missingness Type,Missing Rate,Feature Group Blanked,Model,Random Seed,Test Set,AUROC,AUPRC,Brier Score
0,Baseline,None,0.0,None,0.0,None,Logistic Regression,42,In-domain Test,0.845117,0.767814,0.156161
1,Baseline,None,0.0,None,0.0,None,Logistic Regression,42,Shifted Test,0.855254,0.785536,0.150670
2,Baseline,None,0.0,None,0.0,None,LightGBM,42,In-domain Test,0.904079,0.862035,0.123450
3,Baseline,None,0.0,None,0.0,None,LightGBM,42,Shifted Test,0.910867,0.875319,0.119342
4,Baseline,None,0.0,None,0.0,None,Logistic Regression,43,In-domain Test,0.844153,0.766521,0.156136


In [18]:
# Compute generalization gap (Shifted - In-domain) per run and append
gap_cols = ['AUROC', 'AUPRC', 'Brier Score']
group_keys = ['Experiment', 'Condition', 'Noise Rate', 'Missingness Type',
              'Missing Rate', 'Feature Group Blanked', 'Model', 'Random Seed']
gap_rows = []

for _, group in results_df.groupby(group_keys):
    id_row = group[group['Test Set'] == 'In-domain Test']
    sh_row = group[group['Test Set'] == 'Shifted Test']
    if not id_row.empty and not sh_row.empty:
        gap = id_row.iloc[0].to_dict()
        gap['Test Set'] = 'Generalization Gap (Shifted - In-domain)'
        for col in gap_cols:
            gap[col] = sh_row.iloc[0][col] - id_row.iloc[0][col]
        gap_rows.append(gap)

results_df = pd.concat([results_df, pd.DataFrame(gap_rows)], ignore_index=True)
print(f"Results shape (with gaps): {results_df.shape}")

Results shape (with gaps): (252, 12)


In [19]:
summary_df = results_df.groupby(
    ['Experiment', 'Condition', 'Noise Rate', 'Missingness Type',
     'Missing Rate', 'Feature Group Blanked', 'Model', 'Test Set']
).agg(
    mean_AUROC=('AUROC', 'mean'),            std_AUROC=('AUROC', 'std'),
    mean_AUPRC=('AUPRC', 'mean'),            std_AUPRC=('AUPRC', 'std'),
    mean_Brier_Score=('Brier Score', 'mean'), std_Brier_Score=('Brier Score', 'std')
).reset_index()

print(f"Summary shape: {summary_df.shape}")
summary_df.head()

Summary shape: (84, 14)


,Experiment,Condition,Noise Rate,Missingness Type,Missing Rate,Feature Group Blanked,Model,Test Set,mean_AUROC,std_AUROC,mean_AUPRC,std_AUPRC,mean_Brier_Score,std_Brier_Score
0,Baseline,None,0.0,None,0.0,None,LightGBM,Generalization Gap (Shifted - In-domain),0.007086,0.001115,0.013134,0.000965,-0.004137,0.000733
1,Baseline,None,0.0,None,0.0,None,LightGBM,In-domain Test,0.903681,0.001213,0.861985,0.001149,0.123531,0.000752
2,Baseline,None,0.0,None,0.0,None,LightGBM,Shifted Test,0.910767,0.000117,0.875118,0.000259,0.119394,0.000048
3,Baseline,None,0.0,None,0.0,None,Logistic Regression,Generalization Gap (Shifted - In-domain),0.010831,0.000625,0.018366,0.000890,-0.005623,0.000364
4,Baseline,None,0.0,None,0.0,None,Logistic Regression,In-domain Test,0.844291,0.000767,0.767306,0.000690,0.156419,0.000468


## Save Results

In [20]:
results_df.to_csv(os.path.join(output_dir, 'acs_all_experiments_raw_results.csv'), index=False)
summary_df.to_csv(os.path.join(output_dir, 'acs_all_experiments_summary.csv'), index=False)
print(f"Saved to {output_dir}")

Saved to ./thesis_figures_acs/


## Correlation Between In-domain and Shifted Performance

Tests whether in-domain performance (CA) is a reliable proxy for shifted performance (TX) across stressor conditions.
High Spearman/Pearson correlation → synthetic stress tests are a useful proxy for real geographic shift.

In [21]:
def compute_rq3_correlations(results_df, experiment_name, condition_col='Condition'):
    """Compute Spearman and Pearson correlations between in-domain and shifted metrics."""
    exp_df = results_df[results_df['Experiment'] == experiment_name].copy()
    group_keys = ['Model', 'Random Seed', condition_col]

    rows = []
    for model in exp_df['Model'].unique():
        mdf = exp_df[exp_df['Model'] == model]
        for metric in ['AUROC', 'AUPRC', 'Brier Score']:
            in_vals, sh_vals = [], []
            for cond in mdf[condition_col].unique():
                cdf = mdf[mdf[condition_col] == cond]
                id_rows = cdf[cdf['Test Set'] == 'In-domain Test'][metric].values
                sh_rows = cdf[cdf['Test Set'] == 'Shifted Test'][metric].values
                n = min(len(id_rows), len(sh_rows))
                in_vals.extend(id_rows[:n].tolist())
                sh_vals.extend(sh_rows[:n].tolist())

            if len(in_vals) >= 3:
                s_in = pd.Series(in_vals)
                s_sh = pd.Series(sh_vals)
                spearman = s_in.rank(method='average').corr(s_sh.rank(method='average'))
                pearson = s_in.corr(s_sh)
            else:
                spearman, pearson = np.nan, np.nan

            rows.append({
                'Experiment': experiment_name,
                'Model': model,
                'Metric': metric,
                'n_conditions': len(in_vals),
                'Spearman': spearman,
                'Pearson': pearson
            })

    return pd.DataFrame(rows)


rq3_parts = []
for exp_name, cond_col in [
    ('Label Noise',    'Noise Rate'),
    ('Missingness',    'Missingness Type'),
    ('Feature Shift',  'Feature Group Blanked'),
]:
    corr_df = compute_rq3_correlations(results_df, exp_name, cond_col)
    print(f"\n=== {exp_name} ===")
    print(corr_df.to_string(index=False))
    rq3_parts.append(corr_df)

rq3_df = pd.concat(rq3_parts, ignore_index=True)
rq3_df.to_csv(os.path.join(output_dir, 'acs_rq3_correlations.csv'), index=False)
print(f"\nRQ3 correlations saved to {output_dir}")


=== Label Noise ===
 Experiment               Model      Metric  n_conditions  Spearman  Pearson
Label Noise Logistic Regression       AUROC            12  0.944056 0.987271
Label Noise Logistic Regression       AUPRC            12  0.769231 0.790960
Label Noise Logistic Regression Brier Score            12  0.944056 0.999586
Label Noise            LightGBM       AUROC            12  0.706294 0.696765
Label Noise            LightGBM       AUPRC            12  0.881119 0.781421
Label Noise            LightGBM Brier Score            12  0.937063 0.998212

=== Missingness ===
 Experiment               Model      Metric  n_conditions  Spearman  Pearson
Missingness Logistic Regression       AUROC            18  0.720497 0.627582
Missingness Logistic Regression       AUPRC            18  0.896480 0.970245
Missingness Logistic Regression Brier Score            18  0.763975 0.774841
Missingness            LightGBM       AUROC            18  0.629400 0.551127
Missingness            LightGBM   

## Generate Plots

- `plot_baseline_bar_chart`: grouped bar chart of In-domain vs. Shifted performance per model
- `plot_metrics_line_chart`: mean metric vs. condition with ±1 std band
- `plot_gap_line_chart`: generalization gap (Shifted − In-domain) vs. condition
- `plot_gap_bar_chart`: bar chart of generalization gap by feature group

In [22]:
def _mean_std_cols(metric):
    """Return summary_df column names for a given metric."""
    key = metric.replace(' ', '_')
    return f'mean_{key}', f'std_{key}'


def plot_baseline_bar_chart(summary_df, output_dir, file_prefix='acs_baseline'):
    """Grouped bar chart: In-domain vs Shifted performance per model."""
    plot_df = summary_df[
        (summary_df['Experiment'] == 'Baseline') &
        (summary_df['Test Set'].isin(['In-domain Test', 'Shifted Test']))
    ].copy()

    test_sets = ['In-domain Test', 'Shifted Test']
    model_names = plot_df['Model'].unique()
    x = np.arange(len(model_names))
    width = 0.35
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    for metric in metrics:
        mean_col, std_col = _mean_std_cols(metric)
        fig, ax = plt.subplots(figsize=(9, 6))
        for i, ts in enumerate(test_sets):
            ts_df = plot_df[plot_df['Test Set'] == ts]
            means = [ts_df[ts_df['Model'] == m][mean_col].values[0]
                     if not ts_df[ts_df['Model'] == m].empty else 0 for m in model_names]
            stds  = [ts_df[ts_df['Model'] == m][std_col].fillna(0).values[0]
                     if not ts_df[ts_df['Model'] == m].empty else 0 for m in model_names]
            ax.bar(x + i * width, means, width, yerr=stds, capsize=5, label=ts, color=colors[i])
        ax.set_title(f'ACS Baseline {metric}: In-domain vs Shifted', fontsize=14)
        ax.set_xlabel('Model', fontsize=12)
        ax.set_ylabel(metric, fontsize=12)
        ax.set_xticks(x + width / 2)
        ax.set_xticklabels(model_names)
        ax.legend(title='Test Set')
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{file_prefix}_{metric.lower().replace(" ", "_")}.png'), bbox_inches='tight')
        plt.close()
    print('Baseline bar charts saved.')


def plot_metrics_line_chart(summary_df, experiment_name, x_axis_column, hue_column, output_dir, file_prefix):
    """Line chart: mean metric vs. x_axis_column, split by model and test set."""
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    for metric in metrics:
        mean_col, std_col = _mean_std_cols(metric)
        fig, ax = plt.subplots(figsize=(14, 8))
        plot_df = summary_df[
            (summary_df['Experiment'] == experiment_name) &
            (summary_df['Test Set'].isin(['In-domain Test', 'Shifted Test']))
        ].copy().sort_values(by=x_axis_column)
        plot_df[std_col] = plot_df[std_col].fillna(0)

        ci = 0
        for model in plot_df['Model'].unique():
            for hue_val in plot_df[hue_column].unique():
                for ts in ['In-domain Test', 'Shifted Test']:
                    d = plot_df[(plot_df['Model'] == model) &
                                (plot_df[hue_column] == hue_val) &
                                (plot_df['Test Set'] == ts)]
                    if d.empty:
                        continue
                    c = colors[ci % len(colors)]
                    ax.plot(d[x_axis_column], d[mean_col], marker='o', linewidth=2, color=c,
                            label=f'{model} - {hue_val} - {ts}')
                    ax.fill_between(d[x_axis_column],
                                    d[mean_col] - d[std_col], d[mean_col] + d[std_col],
                                    alpha=0.1, color=c)
                    ci += 1

        ax.set_title(f'ACS Mean {metric} vs. {x_axis_column} — {experiment_name}', fontsize=16)
        ax.set_xlabel(x_axis_column, fontsize=12)
        ax.set_ylabel(f'Mean {metric}', fontsize=12)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{file_prefix}_{metric.lower().replace(" ", "_")}_mean.png'), bbox_inches='tight')
        plt.close()
    print(f'Line charts saved: {experiment_name}')


def plot_gap_line_chart(summary_df, experiment_name, x_axis_column, hue_column, output_dir, file_prefix):
    """Line chart: generalization gap vs. x_axis_column."""
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    for metric in metrics:
        mean_col, std_col = _mean_std_cols(metric)
        fig, ax = plt.subplots(figsize=(14, 8))
        plot_df = summary_df[
            (summary_df['Experiment'] == experiment_name) &
            (summary_df['Test Set'] == 'Generalization Gap (Shifted - In-domain)')
        ].copy().sort_values(by=x_axis_column)

        if plot_df.empty:
            print(f'No gap data for {experiment_name} / {metric}.')
            plt.close()
            continue

        plot_df[std_col] = plot_df[std_col].fillna(0)
        ci = 0
        for model in plot_df['Model'].unique():
            for hue_val in plot_df[hue_column].unique():
                d = plot_df[(plot_df['Model'] == model) & (plot_df[hue_column] == hue_val)]
                if d.empty:
                    continue
                c = colors[ci % len(colors)]
                ax.plot(d[x_axis_column], d[mean_col], marker='o', linewidth=2, color=c,
                        label=f'{model} - {hue_val}')
                ax.fill_between(d[x_axis_column],
                                d[mean_col] - d[std_col], d[mean_col] + d[std_col],
                                alpha=0.1, color=c)
                ci += 1

        ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
        ax.set_title(f'ACS Generalization Gap ({metric}) vs. {x_axis_column} — {experiment_name}', fontsize=16)
        ax.set_xlabel(x_axis_column, fontsize=12)
        ax.set_ylabel(f'{metric} Gap (Shifted - In-domain)', fontsize=12)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{file_prefix}_{metric.lower().replace(" ", "_")}_gap.png'), bbox_inches='tight')
        plt.close()
    print(f'Gap line charts saved: {experiment_name}')


def plot_gap_bar_chart(summary_df, experiment_name, category_column, output_dir, file_prefix):
    """Bar chart: generalization gap by category."""
    for metric in metrics:
        mean_col, std_col = _mean_std_cols(metric)
        fig, ax = plt.subplots(figsize=(14, 8))
        plot_df = summary_df[
            (summary_df['Experiment'] == experiment_name) &
            (summary_df['Test Set'] == 'Generalization Gap (Shifted - In-domain)')
        ].copy().sort_values(by=category_column)

        if plot_df.empty:
            print(f'No data for {experiment_name} gap bar chart / {metric}.')
            plt.close()
            continue

        plot_df[std_col] = plot_df[std_col].fillna(0)
        models = plot_df['Model'].unique()
        categories = plot_df[category_column].unique()
        x = np.arange(len(categories))
        width = 0.35

        for i, model_name in enumerate(models):
            md_df = plot_df[plot_df['Model'] == model_name]
            means = [md_df[md_df[category_column] == c][mean_col].values[0]
                     if not md_df[md_df[category_column] == c].empty else 0 for c in categories]
            stds  = [md_df[md_df[category_column] == c][std_col].values[0]
                     if not md_df[md_df[category_column] == c].empty else 0 for c in categories]
            ax.bar(x + i * width, means, width, yerr=stds, capsize=5, label=model_name)

        ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
        ax.set_title(f'ACS {metric} Gap by {category_column} — {experiment_name}', fontsize=16)
        ax.set_xlabel(category_column, fontsize=12)
        ax.set_ylabel(f'{metric} Gap', fontsize=12)
        ax.set_xticks(x + width / 2 * (len(models) - 1))
        ax.set_xticklabels(categories, rotation=45, ha='right')
        ax.legend(title='Model')
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{file_prefix}_{metric.lower().replace(" ", "_")}_gap_bar.png'), bbox_inches='tight')
        plt.close()
    print(f'Bar charts saved: {experiment_name}')


print("Plot functions defined.")

Plot functions defined.


In [ ]:
print("--- Generating Plots ---")

plot_baseline_bar_chart(summary_df, output_dir)
plot_metrics_line_chart(summary_df, 'Label Noise', 'Noise Rate', 'Model', output_dir, 'acs_label_noise')
plot_gap_line_chart(summary_df, 'Label Noise', 'Noise Rate', 'Model', output_dir, 'acs_label_noise')

plot_metrics_line_chart(summary_df, 'Missingness', 'Missing Rate', 'Missingness Type', output_dir, 'acs_missingness')
plot_gap_line_chart(summary_df, 'Missingness', 'Missing Rate', 'Missingness Type', output_dir, 'acs_missingness')

baseline_gap_df = summary_df[
    (summary_df['Experiment'] == 'Baseline') &
    (summary_df['Test Set'] == 'Generalization Gap (Shifted - In-domain)')
][['Model', 'mean_AUROC', 'std_AUROC', 'mean_AUPRC', 'std_AUPRC',
   'mean_Brier_Score', 'std_Brier_Score']].copy()
baseline_gap_df['Feature Group Blanked'] = 'Baseline (No Blanking)'
baseline_gap_df['Experiment'] = 'Feature Shift'
baseline_gap_df['Condition'] = 'Baseline'
baseline_gap_df['Noise Rate'] = 0.0
baseline_gap_df['Missingness Type'] = 'None'
baseline_gap_df['Missing Rate'] = 0.0
baseline_gap_df['Test Set'] = 'Generalization Gap (Shifted - In-domain)'

feature_shift_plot_df = pd.concat([
    summary_df[(summary_df['Experiment'] == 'Feature Shift') &
               (summary_df['Test Set'] == 'Generalization Gap (Shifted - In-domain)')],
    baseline_gap_df
], ignore_index=True)

plot_gap_bar_chart(feature_shift_plot_df, 'Feature Shift', 'Feature Group Blanked', output_dir, 'acs_feature_shift')

print("All plots saved to", output_dir)

--- Generating Plots ---
Baseline bar charts saved.
Line charts saved: Label Noise
Gap line charts saved: Label Noise
Line charts saved: Missingness
Gap line charts saved: Missingness
Bar charts saved: Feature Shift
All plots saved to ./thesis_figures_acs/
